# DR5 Prediction

## Dataset

In [ ]:
from utils import make_train_val_datasets,format_time
import time
import numpy as np
import torch

data_folder = r".\data\P\train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

start_time = time.time()
train_ds, val_ds, global_stats = make_train_val_datasets(
    data_folder, 
    num_workers=1, 
    step_size=10, 
    window_size=30
)
print(f"Dataset Time: {format_time(time.time() - start_time)}")

# Do not uncomment this during review, as it will overwrite the file.
# np.savez("global_stats_P.npz", **global_stats)

lumped_size = train_ds[0]['lumped'].shape[1]
point_size = train_ds[0]['point'].shape[0]
print('lumped_size: ', lumped_size)
print('point_size: ', point_size)


## Training

In [ ]:
from utils import BatteryMFT, train_battery_model
import time

model = BatteryMFT(
    lumped_size=lumped_size, 
    point_size=point_size, 
    hidden_size=256, 
    encoder_method='lstm'   # best_method
).to(device)

start_train = time.time()

model = train_battery_model(
    model=model,
    save_dir=r"./model_demo",
    train_dataset=train_ds,
    val_dataset=val_ds,
    mode='dr5_prediction',
    batch_size=64, 
    epochs=100,
    lr=1e-3, 
    device=device,
    patience=20,
    lambda_recon=0.1,
    lambda_soc=0.5
)

train_time = time.time() - start_train
print(f"Training Time: {format_time(train_time)}")


# Transfer Learning

## PFS

In [ ]:
from utils import BatteryMFT, format_time, make_train_val_datasets, train_battery_model_TL
import time
import numpy as np
import torch

stats = np.load("global_stats_P.npz", allow_pickle=True)
data_folder = r"data\R\train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

start_time = time.time()
train_ds_tl, val_ds_tl, global_stats = make_train_val_datasets(
    folder_dir=data_folder, 
    global_stats=stats, 
    step_size=10,
    num_workers=1
)
data_time = time.time() - start_time
print(f"Dataset Time: {format_time(data_time)}")

print("="*8 + " Transfer: PFS " + "="*8)
model = BatteryMFT(
    lumped_size=8,
    point_size=21,
    hidden_size=256,
    encoder_method='lstm'
)
checkpoint = torch.load(r".\model\best_dr5_prediction.pth")
model.load_state_dict(checkpoint)

print("❄️ Freezing feature extraction layers...")
for name, param in model.named_parameters():
    if any(key in name for key in ["lumped_encoder", "recon_head"]):
        param.requires_grad = False
    else:
        param.requires_grad = True
        print(f"🔥 Active: {name}")

start_infer = time.time()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), 
                              lr=5e-4, weight_decay=1e-4)

model_final = train_battery_model_TL(
    model, 
    save_dir=r"./model_TL_demo/R_PFS", 
    train_dataset=train_ds_tl, 
    val_dataset=val_ds_tl, 
    optimizer=optimizer,
    batch_size=64, 
    epochs=100, 
    patience=15, 
    device=device,
)

infer_time = time.time() - start_infer
print(f"Training Time: {format_time(infer_time)}")

## LLR

In [ ]:
from utils import BatteryMFT, format_time, make_train_val_datasets, train_battery_model_TL
import time
import numpy as np
import torch

stats = np.load("global_stats_P.npz", allow_pickle=True)
data_folder = r"data\R\train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

start_time = time.time()
train_ds_tl, val_ds_tl, global_stats = make_train_val_datasets(
    folder_dir=data_folder, 
    global_stats=stats, 
    step_size=10,
    num_workers=1
)
data_time = time.time() - start_time
print(f"Dataset Time: {format_time(data_time)}")

print("="*8 + " Transfer: LLR " + "="*8)
model = BatteryMFT(
    lumped_size=8,
    point_size=21,
    hidden_size=256,
    encoder_method='lstm'
)
checkpoint = torch.load(r".\model\best_dr5_prediction.pth")
model.load_state_dict(checkpoint)

start_infer = time.time()
optimizer = torch.optim.Adam([
    {'params': model.lumped_encoder.parameters(), 'lr': 1e-5},
    {'params': model.soc_rnn.parameters(), 'lr': 1e-4},
    {'params': model.dr5_head.parameters(), 'lr': 1e-3},
])

model_final = train_battery_model_TL(
    model, 
    save_dir=r"./model_TL_demo/R_LLR", 
    train_dataset=train_ds_tl, 
    val_dataset=val_ds_tl, 
    optimizer=optimizer,
    batch_size=64, 
    epochs=100, 
    patience=15, 
    device=device,
)

infer_time = time.time() - start_infer
print(f"Training Time: {format_time(infer_time)}")

# Test

## E2E Model

### Single Discharge Result

In [ ]:
from utils import BatteryMFT, evaluate_multitask, make_single_test_dataset
import numpy as np
import torch

file_path = r".\data\P\test\discharge_segment_P218\dis_seg_113_1.pkl"
model_path = r".\model\best_dr5_prediction.pth"
save_dir = r"./results"

# ==============================================
stats = np.load("global_stats_P.npz", allow_pickle=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = torch.load(model_path)

test_dataset_seg = make_single_test_dataset(
    file_path=file_path, 
    global_stats=stats,
    step_size=1,
)

model = BatteryMFT(
    lumped_size=8,
    point_size=21,
    hidden_size=256,
    encoder_method='lstm'
)
model.load_state_dict(model_path)
model.to(device)

evaluate_multitask(
    model, test_dataset_seg, device, stats,
    save_dir=save_dir
    )


### Fig.4 Single Discharge

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df_path = r".\results\result_multitask_all.pkl"
df_seg = pd.read_pickle(df_path)

ftsz = 20
tksz = 16

# ================= Voltage Reconstruction ========================
n = len(df_seg)
win_sel = int(n/2)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
# cmap
c1 = plt.get_cmap('Blues')(np.linspace(0.3, 0.6, n))
c2 = plt.get_cmap('Reds')(np.linspace(0.3, 0.6, n))

for i in range(n):
    df_win = df_seg.iloc[i]
    tau = range(len(df_win['v_label']))

    ax.plot(tau, i+1, df_win['v_recon'], '-', color=c1[i], alpha=0.5, linewidth=2)
    ax.plot(tau, i+1, df_win['v_label'], '--', color=c2[i], alpha=0.5, linewidth=1)

# --------------------------------------------------------
win_idxs = [0, win_sel, n-1]
for idx in win_idxs:
    df_win = df_seg.iloc[idx]
    tau = range(len(df_win['v_label']))
    ax.plot(tau, idx+1, df_win['v_recon'], '-', color="#3C86F5", alpha=1, linewidth=3)
    ax.plot(tau, idx+1, df_win['v_label'], '--', color="r", alpha=1, linewidth=1.5)

# --------------------------------------------------------
ax.set_xlabel(r'$\tau$ (s)', fontsize=ftsz, labelpad=10)
ax.set_ylabel(r'Window Index', fontsize=ftsz, labelpad=25)
ax.set_zlabel(r'Voltage (V)', fontsize=ftsz, labelpad=10)
ax.tick_params(axis='both', which='major', labelsize=tksz)
ax.tick_params(axis='y',rotation=20)
ax.view_init(elev=10, azim=45)
plt.tight_layout()
plt.show()

# ================= Voltage Prediction ========================
pred_last = df_seg['v_recon'].apply(lambda x: x[-1])
gt_last = df_seg['v_label'].apply(lambda x: x[-1])
recon_res = pred_last - gt_last
win_idx = range(len(pred_last))

color_pred = "#3C86F5"
color_meas = "#F06F6F"
color_res = "#A763DB"

fig, ax = plt.subplots(1, 1, figsize=(5, 2.5), sharex=True)
ax1 = ax.twinx()
ax1.fill_between(win_idx, 0, recon_res, label='Residual', color=color_res, alpha=0.1)
ax1.plot(win_idx, recon_res, color=color_res, linewidth=1.5, alpha=0.1)
ax1.axhline(0, color='black', linewidth=1, linestyle=':', alpha=0.5) # 画一条 0 刻度线
ax.plot(win_idx, pred_last, color=color_pred, lw=3, ls='-', label='Reconstructed')
ax.plot(win_idx, gt_last, lw=1.5, ls='--', color=color_meas, label='Measured')
ax.set_xlabel(r'Drving Time ($\times10$ s)', fontsize=14)
ax.set_ylabel('Voltage (V)', fontsize=14)
ax.tick_params(labelsize=12)
ax1.set_ylabel('Residual (V)', color=color_res, fontsize=14)
ax1.tick_params(axis='y', labelcolor=color_res, labelsize=12)
ax.legend(fontsize=12, frameon=False, loc='best')
plt.show()

# ================= SOC Estimation ========================
pred = df_seg['soc_pred']
gt = df_seg['soc_label']
res = pred - gt
win_idx = range(len(pred))

color_pred = "#E78738"
color_meas = "#A80000"
color_res = "#7E7E7E"

fig, ax = plt.subplots(1, 1, figsize=(5, 2.5), sharex=True)
ax1 = ax.twinx()
ax1.fill_between(win_idx, 0, res, label='Residual', color=color_res, alpha=0.1)
ax1.plot(win_idx, res, color=color_res, linewidth=1.5, alpha=0.1)
ax1.axhline(0, color='black', linewidth=1, linestyle=':', alpha=0.5) # 画一条 0 刻度线
ax.plot(win_idx, pred, color=color_pred, lw=3, ls='-', label='Predicted')
ax.plot(win_idx, gt, lw=1.5, ls='--', color=color_meas, label='Label')
ax.set_xlabel(r'Drving Time ($\times10$ s)', fontsize=14)
ax.set_ylabel('SOC (%)', fontsize=14)
ax.tick_params(labelsize=12)
ax1.set_ylabel('Residual (%)', color=color_res, fontsize=14)
ax1.tick_params(axis='y', labelcolor=color_res, labelsize=12)
ax.legend(fontsize=12, frameon=False, loc='best')
plt.show()

# ================= DR5 Prediction ========================
pred = df_seg['dr5_pred']
gt = df_seg['dr5_label']
res = pred - gt
win_idx = range(len(pred))

color_pred = "#B36BEA"
color_meas = "#A80000"
color_res = "#7E7E7E"

fig, ax = plt.subplots(1, 1, figsize=(5, 2.5), sharex=True)
ax1 = ax.twinx()
ax1.fill_between(win_idx, 0, res, label='Residual', color=color_res, alpha=0.1)
ax1.plot(win_idx, res, color=color_res, linewidth=1.5, alpha=0.1)
ax1.axhline(0, color='black', linewidth=1, linestyle=':', alpha=0.5) # 画一条 0 刻度线
ax.plot(win_idx, pred, color=color_pred, lw=3, ls='-', label='Predicted')
ax.plot(win_idx, gt, lw=1.5, ls='--', color=color_meas, label='Label')
ax.set_xlabel(r'Drving Time ($\times10$ s)', fontsize=14)
ax.set_ylabel('DR5 (km)', fontsize=14)
ax.tick_params(labelsize=12)
ax1.set_ylabel('Residual (km)', color=color_res, fontsize=14)
ax1.tick_params(axis='y', labelcolor=color_res, labelsize=12)
ax.legend(fontsize=12, frameon=False, loc='best')
plt.show()



## TL

### Single Discharge Result

In [ ]:
from utils import BatteryMFT, evaluate_multitask, make_single_test_dataset
import numpy as np
import torch

# ============ Data & Model =============
file_path = r".\data\R\test\discharge_segment_R005\dis_seg_1021_5.pkl"
stats = np.load("global_stats_P.npz", allow_pickle=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
test_dataset_seg = make_single_test_dataset(
    file_path=file_path, 
    global_stats=stats,
    step_size=1,
)
model = BatteryMFT(
    lumped_size=8,
    point_size=21,
    hidden_size=256,
    encoder_method='lstm'
)

# ============== Base ===============
model_path = r".\model\best_dr5_prediction.pth"
save_dir = r"./results_TL/R_Base"
model_path = torch.load(model_path)
model.load_state_dict(model_path)
model.to(device)

evaluate_multitask(
    model, test_dataset_seg, device, stats,
    save_dir=save_dir
    )

# ============== TL ===============
TL_lst = ['PFS','LLR']
for strategy in TL_lst:
    model_path = rf"./model/R_{strategy}\best_TL.pth"
    save_dir = rf"./results_TL/R_{strategy}"
    model_path = torch.load(model_path)
    model.load_state_dict(model_path)
    model.to(device)
    evaluate_multitask(
        model, test_dataset_seg, device, stats,
        save_dir=save_dir
        )


### Fig.5 Single Discharge

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

model_names = ['Base', 'LLR', 'PFS']
colors = ["#C4BAF0", "#A7A359", "#6BC2C9"]

plt.figure(figsize=(4.5, 2.5))

for i in range(len(model_names)):
    df_path = rf".\results_TL\R_{model_names[i]}\result_multitask_all.pkl"
    df_seg = pd.read_pickle(df_path)
    win_idx = range(len(df_seg['dr5_pred']))
    plt.plot(win_idx, df_seg['dr5_pred'], color=colors[i], lw=1.5, ls='-', 
             label=model_names[i], alpha=0.8)

plt.plot(win_idx, df_seg['dr5_label'], color='r', lw=1, ls='--', label='Label')

plt.xlabel(r'Driving Time ($\times 10$s)', fontsize=14)
plt.ylabel('DR5 (km)', fontsize=14)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.legend(fontsize=11, frameon=False, loc='lower center', bbox_to_anchor=(0.45, 1), ncol=4)
plt.show()